# EQO notebook: ChatQEC tool gallery

This notebook exercises all eight tools from the canonical `chatqec-mcp-tools` source as admitted EQO workflows. Each run goes through EQO's API, virtual Slurm worker, typed artifacts, and pinned OCI image. It never starts the MCP server or invokes a tool directly from the notebook.

Start EQO Local first. The optional ChatQEC tool images must be installed with the identities reported by `eqo local diagnose`; the default internal development profile validates them before accepting these examples.

In [ ]:
import json
import os

from eqo import EQOClient, render_artifact, render_run

EQO_ENDPOINT = os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080")
eqo = EQOClient.connect(EQO_ENDPOINT)
eqo.health()

In [ ]:
published = {item["id"]: item for item in eqo.workflows.list()}

def run_example(workflow_id, *, inputs=None, timeout=600):
    workflow = published.get(workflow_id)
    if workflow is None:
        raise RuntimeError(f"{workflow_id} is not published. Restart EQO Local after updating the profile.")
    run = eqo.workflows.submit(
        workflow["id"],
        workflow["version"],
        inputs=inputs,
        execution_target="development-slurm-docker",
    )
    completed = run.wait(timeout=timeout)
    if completed.state != "succeeded":
        raise RuntimeError(f"{workflow_id} ended in {completed.state}; inspect render_run(completed).")
    return completed

[name for name in published if name.startswith("chatqec-")]

## Code parameters

This runs `code-params` for a rotated surface code and returns the typed `qhpc.qec-code-parameters@1` result.

In [ ]:
code_parameters = run_example("chatqec-code-parameters")
render_artifact(code_parameters.artifacts.by_type("qhpc.qec-code-parameters@1"))

## One typed circuit, four tools

`qec-circuit-build` uses the pinned LightStim source to produce a detector-annotated Stim artifact. The same artifact is then passed to Stim simulation, SVG diagram generation, and PyMatching decoding. This is the main tool-composition example.

In [ ]:
toolchain = run_example("chatqec-qec-toolchain")
render_run(toolchain)

In [ ]:
for artifact_type in (
    "qhpc.qec-circuit-build-report@1",
    "qhpc.stim-simulation-samples@1",
    "qhpc.stim-diagram@1",
    "qhpc.qec-decoder-result@1",
):
    display(render_artifact(toolchain.artifacts.by_type(artifact_type)))

The SVG is intentionally summarized by the notebook helper. Download the checksum-verified `qhpc.stim-diagram@1` artifact if you need to inspect it; notebook rendering never grants an artifact browser privileges.

## Threshold-sweep evidence

The workflow returns empirical error-rate rows and a sanitized plot. It does not infer or claim a threshold.

In [ ]:
sweep = run_example("chatqec-threshold-sweep")
display(render_artifact(sweep.artifacts.by_type("qhpc.qec-threshold-sweep@1")))
display(render_artifact(sweep.artifacts.by_type("qhpc.qec-threshold-plot@1")))

## Non-Clifford Tsim simulation

Create the typed Tsim input from the canonical T-gate detector fixture, then submit it to the separate pinned Tsim OCI image.

In [ ]:
tsim_circuit = """RX 0\nT 0\nH 0\nM 0\nDETECTOR rec[-1]\n"""
tsim_input = eqo.artifacts.create_input(
    "qhpc.tsim-circuit@1", tsim_circuit, name="t-gate-detector.tsim"
)
tsim = run_example("chatqec-tsim-simulation", inputs={"circuit": tsim_input.id})
render_artifact(tsim.artifacts.by_type("qhpc.tsim-simulation-samples@1"))

## Gemini Logical Circuit Builder link

The generator is offline and deterministic. Opening its resulting URL is deliberately left to the user because it discloses the circuit to an external service.

In [ ]:
glcb_specification = {
    "qubits": 2,
    "gates": [
        {"gate": "h", "column": 0, "targets": [0]},
        {"gate": "cx", "column": 1, "targets": [0, 1]},
    ],
    "sampler": "clifft",
}
glcb_input = eqo.artifacts.create_input(
    "qhpc.glcb-circuit-spec@1",
    json.dumps(glcb_specification),
    name="bell.glcb.json",
)
glcb = run_example("chatqec-glcb-link", inputs={"circuit": glcb_input.id})
render_artifact(glcb.artifacts.by_type("qhpc.glcb-visualization-link@1"))